# Lab 14 · Thực hành kể chuyện: kim tự tháp, lời đúng mức và thẩm định

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 14**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook bài giảng Bài 14 thẩm định một báo cáo do AI tạo, gồm năm kết luận. Trong bài thực hành,
bạn sẽ tự viết kết luận đúng mức, tạo một ví dụ về nghịch lý Simpson và thẩm định một kết luận mới.
Tuần này cũng là **tuần nộp bài tập lớn**; 30 phút cuối dành để kiểm tra trước khi nộp.

**Mục tiêu bài lab**

1. Sắp xếp một mục báo cáo theo cấu trúc kim tự tháp (kết luận → bằng chứng → giới hạn).
2. Phân biệt cách diễn đạt đúng mức và quá mức trên các cặp số liệu–kết luận cụ thể.
3. Tự tính một nghịch lý Simpson: hai nhóm cùng tăng, tổng gộp vẫn giảm.
4. Thẩm định một kết luận mới bằng quy trình bốn bước.

Phiên bản đề: `storytelling-review-v1` (25/09/2026). Lưu đúng tên **`lab-14.ipynb`**.

## Cách làm và nộp bài

- Với phần có hướng dẫn, hãy **tự gõ, không dùng AI**. Đây là chế độ 🚫 đóng, tương ứng với các bài kiểm tra định kỳ ở giờ lý thuyết.
  Bài tự làm cuối lab ở chế độ ✅ mở: được dùng AI nhưng phải khai báo theo chính sách của môn học.
- Bài 1, 2, 6 là **câu trả lời gán vào biến** (`thu_tu`, `nhan`, `phan_quyet`) — grader đọc giá trị biến.
  Public check chỉ kiểm **định dạng**, không cho biết đúng/sai; tự lập luận trước khi xem phần diễn giải của giảng viên.
- Bài 3, 4, 5 là **hàm**: grader chạy riêng cell đó với bảng khác (tên cột, số kỳ khác…), không phụ thuộc biến `bang`, `ds` ở cell khác.
  `pd`, `np` được cung cấp sẵn. Giữ tên hàm/biến, tham số, cell ID và metadata; không ghi cứng kết quả.
- Một check lỗi không dừng các check sau. TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1 kim tự tháp 10 · Bài 2 năm cặp đúng mức/quá mức 15 (3 điểm mỗi cặp) | 25 | Tự động |
| Bài 3 giá trung bình gộp 20 · Bài 4 tỷ trọng phân khúc 10 | 30 | Tự động |
| Bài 5 truy số 15 · Bài 6 phán quyết thẩm định 10 | 25 | Tự động |
| Lập luận cho Bài 2, nguyên nhân nghịch lý Simpson, lý do phán quyết | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

In [ ]:
import numpy as np
import pandas as pd

## Bài 1 · Xếp kim tự tháp

Bốn câu dưới đây thuộc một mục báo cáo về giá tại Santiago nhưng đang bị xáo trộn.
Xếp lại theo **kim tự tháp**: (1) kết luận → (2) bằng chứng → (3) giới hạn →
(4) chi tiết phương pháp (đặt ở cuối hoặc trong phụ lục).

### Câu trả lời · 10 điểm tự động

Gán biến `thu_tu` là **list 4 chữ cái** (mỗi chữ trong `CAU` đúng một lần) theo thứ tự kim tự tháp.

In [ ]:
CAU = {
 "A": "Con số dựa trên 17.688 chỗ ở có giá hợp lệ ở mốc chụp 29/06/2026; 846 chỗ ở không có giá đã được loại và ghi trong qa_report.",
 "B": "Giá thuê điển hình ở Santiago là 59.000 CLP/đêm (khoảng 1,65 triệu đồng), thấp hơn mức trung bình 118.000 CLP.",
 "C": "Trung vị ít bị ảnh hưởng bởi giá ngoại lai: 177 giá trị cực đoan đã gắn cờ làm trung bình cao hơn 44% nhưng gần như không làm thay đổi trung vị.",
 "D": "Kết luận chỉ áp dụng cho giá niêm yết một đêm, chưa gồm phí vệ sinh/dịch vụ; giá thực tế khách trả có thể cao hơn.",
}
for k, v in CAU.items():
    print(f"{k}. {v[:100]}…")

In [ ]:
# TODO: điền thứ tự 4 chữ cái theo kim tự tháp
thu_tu = [...]

In [ ]:
def check_q1():
    if ... in thu_tu:
        raise NotImplementedError
    assert isinstance(thu_tu, list) and sorted(thu_tu) == sorted(CAU), "Mỗi chữ A–D xuất hiện đúng một lần"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q1 · định dạng", check_q1)

**Lý do sắp xếp:** câu nào là kết luận, câu nào là bằng chứng, giới hạn, phương pháp? …

## Bài 2 · Lời đúng mức với số liệu

Với mỗi cặp (số liệu → lời viết), gán `"vua"` (đúng mức) hoặc `"qua"` (quá mức).

### Câu trả lời · 15 điểm tự động (3 điểm mỗi cặp)

Gán biến `nhan` là dict `{1: ..., 2: ..., 3: ..., 4: ..., 5: ...}`, mỗi giá trị là `"vua"` hoặc `"qua"`.

In [ ]:
CAP = {
 1: ("So cùng kỳ: +53,7% (T5/2026 so với T5/2025)", "Thị trường tăng trưởng mạnh so với cùng kỳ"),
 2: ("+3,0% so với tháng trước", "Bứt phá thần tốc trong tháng 5"),
 3: ("Hệ số tương quan 0,081", "Nhiệt độ ngày ảnh hưởng rõ rệt tới lượng đánh giá"),
 4: ("Đúng 1 chỗ ở đòi ở tối thiểu 730 đêm", "Một vài ca cực đoan cần gắn cờ"),
 5: ("Nhóm es chiếm 65,4% (quy tắc chưa kiểm định)", "Chắc chắn 2/3 khách nói tiếng Tây Ban Nha"),
}
for k, (so_lieu, loi) in CAP.items():
    print(f"{k}. {so_lieu}  →  \"{loi}\"")

In [ ]:
# TODO: gán nhãn "vua" hoặc "qua" cho từng cặp
nhan = {1: ..., 2: ..., 3: ..., 4: ..., 5: ...}

In [ ]:
def check_q2():
    if ... in nhan.values():
        raise NotImplementedError
    assert set(nhan) == set(CAP), "Cần đúng 5 khóa 1–5"
    assert set(nhan.values()) <= {"vua", "qua"}, "Mỗi nhãn là 'vua' hoặc 'qua'"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q2 · định dạng", check_q2)

**Lập luận:** với mỗi cặp "quá mức", từ nào vượt quá điều số liệu cho thấy, và nên viết lại thế nào? …

## Bài 3 · Tự tạo một nghịch lý Simpson

Hai phân khúc **cùng tăng giá 10%**, nhưng giá trung bình gộp lại giảm. Hãy tính để kiểm tra.
Các số dưới đây chỉ dùng để minh hoạ cơ chế, không phải dữ liệu Santiago.

### Hợp đồng hàm · 20 điểm tự động

`pooled_means(bang: pd.DataFrame, ky: str = "ky", n: str = "n", gia: str = "gia_tb") -> pd.Series`

Nhận bảng mỗi dòng là một (kỳ, phân khúc) với số chỗ ở ở cột `n` và giá trung bình của phân khúc ở cột `gia`
(tên cột truyền qua tham số). Trả Series: index là các kỳ, giá trị là **giá trung bình gộp** của kỳ đó
`tổng(n × gia) / tổng(n)` (trung bình có trọng số, **không** phải trung bình cộng các dòng). Không sửa input, không làm tròn.

In [ ]:
bang = pd.DataFrame({
    "ky":        ["T9",  "T9",  "T6",  "T6"],
    "phan_khuc": ["cao", "re",  "cao", "re"],
    "n":         [100,   100,   80,    220],
    "gia_tb":    [60.0,  30.0,  66.0,  33.0],   # nghìn CLP — mỗi phân khúc +10%
})
bang

In [ ]:
def pooled_means(bang: pd.DataFrame, ky: str = "ky", n: str = "n", gia: str = "gia_tb") -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    gop = pooled_means(bang)
    assert isinstance(gop, pd.Series) and set(gop.index) == {"T6", "T9"}
    assert round(gop["T9"], 1) == 45.0 and round(gop["T6"], 1) == 41.8, "gộp: 45.0 → 41.8"
    assert gop["T6"] != bang.loc[bang["ky"] == "T6", "gia_tb"].mean(), "Cần trung bình có trọng số theo n"

public_check("Q3 · ví dụ công khai", check_q3)

## Bài 4 · Cơ cấu mẫu thay đổi

### Hợp đồng hàm · 10 điểm tự động

`segment_share(bang: pd.DataFrame, phan_khuc: str, ky: str = "ky", nhom: str = "phan_khuc", n: str = "n") -> pd.Series`

Nhận bảng như Bài 3 và tên một phân khúc. Trả Series: index là các kỳ, giá trị là **tỷ trọng số chỗ ở** của phân khúc đó
trong kỳ (tổng `n` của phân khúc / tổng `n` của kỳ, thang 0–1); kỳ không có phân khúc đó thì tỷ trọng là 0. Không sửa input.

In [ ]:
def segment_share(bang: pd.DataFrame, phan_khuc: str, ky: str = "ky", nhom: str = "phan_khuc", n: str = "n") -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    ty_trong = segment_share(bang, "re")
    assert isinstance(ty_trong, pd.Series) and set(ty_trong.index) == {"T6", "T9"}
    assert ty_trong["T9"] == 0.5 and round(ty_trong["T6"], 2) == 0.73
    assert segment_share(bang[bang["phan_khuc"] == "cao"], "re").tolist() == [0.0, 0.0]

public_check("Q4 · ví dụ công khai", check_q4)

In [ ]:
def simpson():
    gop, re = pooled_means(bang), segment_share(bang, "re")
    return (f"Mỗi phân khúc +10%, nhưng gộp: {gop['T9']:.1f} → {gop['T6']:.1f} (GIẢM {(1 - gop['T6'] / gop['T9']):.0%}).\n"
            f"Tỷ trọng phân khúc rẻ: {re['T9']:.0%} → {re['T6']:.0%}.")

preview("Bài 3–4", simpson)

**Nguyên nhân:** vì sao giá gộp giảm dù cả hai phân khúc đều tăng giá? Khi so hai mốc chụp trong bài tập lớn cần kiểm tra gì? …

## Bài 5 · Thẩm định một kết luận mới — bước 1: truy số

Kết luận này chưa có trong notebook bài giảng: *"Chủ nhà chuyên nghiệp **được khách yêu thích hơn**
so với chủ nhà cá nhân — 15,5 so với 12,7 đánh giá/năm."* Hãy thẩm định theo bốn bước, bắt đầu bằng **truy số**.

### Hợp đồng hàm · 15 điểm tự động

`reviews_by_host_type(ds: pd.DataFrame, nguong: int = 5) -> pd.Series`

Nhận bảng chỗ ở có `calculated_host_listings_count` và `number_of_reviews_ltm`. Nhóm theo điều kiện
`calculated_host_listings_count >= nguong` (bool; không thêm cột vào `ds`) và trả Series trung bình `number_of_reviews_ltm`
của từng nhóm: index là các giá trị `False`/`True` có mặt, không làm tròn.
Đối chiếu snapshot: **15,5** (chuyên nghiệp) và **12,7** (cá nhân) đánh giá/năm.

In [ ]:
# 12 chỗ ở GIẢ LẬP cho bước truy số; đặt USE_SNAPSHOT = True để dùng listings.csv của snapshot Santiago.
DEMO_DS = pd.DataFrame({
    "id": range(301, 313),
    "calculated_host_listings_count": [1, 7, 2, 12, 5, 1, 3, 9, 1, 6, 2, 4],
    "number_of_reviews_ltm":          [8, 20, 14, 31, 9, 0, 22, 16, 5, 12, 18, 10],
    "review_scores_rating":           [4.9, 4.6, 4.95, 4.7, 4.5, None, 4.85, 4.6, 4.9, 4.55, 4.8, 4.75],
})
USE_SNAPSHOT = False
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
ds = pd.read_csv(URL) if USE_SNAPSHOT else DEMO_DS.copy()
print("Snapshot Santiago" if USE_SNAPSHOT else "GIẢ LẬP — 12 chỗ ở", "·", len(ds), "dòng")

In [ ]:
def reviews_by_host_type(ds: pd.DataFrame, nguong: int = 5) -> pd.Series:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    kiem = reviews_by_host_type(DEMO_DS)
    assert isinstance(kiem, pd.Series) and set(kiem.index) == {False, True}
    assert round(kiem[True], 1) == 17.6 and round(kiem[False], 1) == 11.0
    assert "chuyen" not in DEMO_DS.columns
    assert round(reviews_by_host_type(DEMO_DS, 10)[True], 1) == 31.0

public_check("Q5 · ví dụ công khai", check_q5)

In [ ]:
preview("Bài 5 — truy số trên dữ liệu đang dùng", lambda: reviews_by_host_type(ds).round(1))

## Bài 6 · Bước 2 (phương pháp) và bước 3 (diễn giải): phán quyết

Chọn **một** phán quyết đúng nhất cho kết luận ở Bài 5:

- **A.** "Giữ nguyên — số đúng thì kết luận đúng."
- **B.** "Sửa cách diễn đạt: số đo là LƯỢNG ĐÁNH GIÁ (biến đại diện cho lượng đặt phòng), không đo 'yêu thích'; muốn nói yêu thích
  phải dùng điểm đánh giá — viết lại thành 'có nhiều đánh giá hơn' và kiểm thêm theo từng nhóm."
- **C.** "Bác bỏ — số liệu sai."

### Câu trả lời · 10 điểm tự động

Gán biến `phan_quyet` là một chữ `"A"`, `"B"` hoặc `"C"`.

In [ ]:
# TODO: chọn MỘT phán quyết
phan_quyet = "..."

In [ ]:
def check_q6():
    if phan_quyet == "...":
        raise NotImplementedError
    assert phan_quyet in {"A", "B", "C"}, "Chọn một chữ A, B hoặc C"
    print("Định dạng hợp lệ — đáp án do grader chấm.")

public_check("Q6 · định dạng", check_q6)

**Lý do phán quyết và bước 4 (kiểm thêm):** vì sao chọn như vậy? Nếu muốn nói về "yêu thích", cần số liệu nào và cần kiểm thêm theo nhóm nào? …

## Bài tự làm ✅ mở · Thẩm định chéo bản nháp của nhóm — không cộng vào 80 điểm tự động

Chọn một mục trong bản nháp báo cáo của nhóm và: (1) thực hiện quy trình thẩm định bốn bước với
**hai kết luận quan trọng nhất**; (2) kiểm tra từng câu bằng tiêu chí "lời đúng mức" ở Bài 2;
(3) đối chiếu mọi nội dung do AI hỗ trợ viết với mục "AI sai ở đâu" trong `AI_USAGE.md`.
Nếu nhóm có dùng AI nhưng mục này còn trống, hãy bổ sung trước khi nộp.

**Ghi chép thẩm định chéo của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu phép tính tay hoặc số liệu đã đối chiếu.

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm theo nhóm, không tính điểm lab)

Hạn nộp: **23:59 Chủ nhật tuần này** (ngày cụ thể: Canvas Portal). Hãy kiểm tra theo danh sách
trong slide Bài 15; dưới đây là bản rút gọn để nhóm tự rà soát:

1. ☐ `run_pipeline.py`/Makefile: **một lệnh** chạy từ dữ liệu gốc đến mọi kết quả; thành phố và
   mốc chụp nằm trong `configs/`, không ghi cứng trong mã.
2. ☐ Bộ nhớ đệm LLM đầy đủ; `--skip-llm` chạy **không cần khoá API**; khoá không xuất hiện
   trong kho mã nguồn, kể cả lịch sử commit (kiểm tra bằng `grep`).
3. ☐ `data/samples/` có ít nhất 100 nhãn tay; `reports/` có PDF và `qa_report.csv`;
   `figures/` sinh tự động.
4. ☐ `AI_USAGE.md` cập nhật lần cuối — mục "AI sai ở đâu" có ví dụ thật.
5. ☐ **Bài kiểm tra máy sạch** đã được lên lịch: một thành viên chạy `git clone` vào thư mục mới và
   làm theo README trước thứ Bảy; gắn thẻ `final` và ghi mã commit vào báo cáo.

> Giảng viên thực hành trao đổi với từng nhóm về các mục chưa hoàn tất và người phụ trách trước
> thứ Bảy; các nhóm có nguy cơ trễ cần được báo lại cho giảng viên sau buổi học.

## Tóm tắt bài thực hành

| Bạn đã làm | Dùng cho |
|---|---|
| Kim tự tháp: kết luận trước, phương pháp sau | mọi mục của báo cáo nộp tuần này |
| Năm cặp số liệu–kết luận: đúng mức hay quá mức | tự kiểm tra từng câu trước khi nộp |
| Tự tính nghịch lý Simpson: cơ cấu nhóm làm số gộp đổi chiều | so sánh hai mốc chụp trong báo cáo |
| Thẩm định "yêu thích" và "có nhiều đánh giá hơn" | chuẩn bị cho các câu hỏi tương tự trong vấn đáp |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Tuần sau là buổi **vấn đáp**. Hãy chuẩn bị theo slide Bài 15 và kế hoạch ôn tập trong đó,
đồng thời hoàn thành việc nộp bài đúng hạn.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-14.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_14.ipynb`).
2. Hoàn thành các câu trả lời, hàm TODO, lập luận và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
4. Tạo tag `submit/lab-14/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-14` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()